# HINTy: Experiments

## Setup DSPy & MLflow

In [1]:
from dotenv import load_dotenv
load_dotenv()

# Configure DSPy to use GPT-5-mini
import dspy
gpt5_mini = dspy.LM("openai/gpt-5-mini")
dspy.configure(lm=gpt5_mini)

import mlflow

# Host local MLflow tracking server
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("HINTy Development")

mlflow.dspy.autolog()

2026/01/06 16:23:42 WARNING mlflow.utils.autologging_utils: MLflow dspy autologging is known to be compatible with 2.5.43 <= dspy, but the installed version is 3.1.0b1. If you encounter errors during autologging, try upgrading / downgrading dspy to a compatible version, or try upgrading MLflow.


## Load MATH dataset

We use the [`hendryks/MATH`](https://github.com/hendrycks/math/) dataset (**M**athematics **A**ptitude **T**est of **H**euristics) to evaluate our DSPy project. The dataset contains high-school level competition math problems, with difficulty ranging from level 1 to 5.

Solutions in this dataset are provided within boxes (i.e. \boxed{...} in LaTeX), so the helper function `extract_boxed` helps extract these solutions.

In [2]:
from datasets import load_dataset
import random
import re

def extract_boxed(text):
    """Extract answer from \boxed{...}"""
    match = re.search(r'\\boxed\{', text)
    if not match:
        return None
    start = match.end()
    depth = 1
    i = start
    while i < len(text) and depth > 0:
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
        i += 1
    return text[start:i-1] if depth == 0 else None


def init_dataset():
    """Load MATH dataset"""
    data = load_dataset("qwedsacf/competition_math")["train"]
    data = [
        dspy.Example({
            "problem": x["problem"],
            "level": x["level"],
            "type": x["type"],
            "solution": x["solution"],
            "answer": extract_boxed(x["solution"]),
        }).with_inputs("problem")
        for x in data
    ]

    random.Random(0).shuffle(data)

    return data

We only take the first 100 entries of this dataset for evaluation (to conserve my credits). Here's an example of an entry of our dataset.

In [3]:
DATA_SIZE = 100
full_dataset = init_dataset()
dataset = full_dataset[:DATA_SIZE]

example = dataset[0]
print("Problem: ", example.problem)
print("Level: ", example.level)
print("Type: ", example.type)
print("Solution: ", example.solution)
print("Answer: ", example.answer)

Problem:  Consider the function $f(x)=5x+4$.  What is $f(1)$?
Level:  Level 1
Type:  Algebra
Solution:  We have $f(1) = 5\cdot 1+4 =5+4=\boxed{9}$.
Answer:  9


## Hint Generation

In HINTy, we use the following DSPy signature & Chain-of-Thought module to generate sequential hints. The signature `MathHintsGenerator` specifies that 
- the input to this DSPy module is the math problem, of type str; and
- the output to this DSPy module is a list of hints, of type list[str].

In [4]:
class MathHintsGenerator(dspy.Signature):
    """Generate a sequence of helpful hints as smaller math questions that guide towards solving the problem. 
    Each successive hint should builds upon the previous hint and get closer to the solution.
    The solution to the final hint should be the same as the solution to the original problem."""

    problem: str = dspy.InputField(desc="The original math problem to solve")
    hints: list[str] = dspy.OutputField(desc="A list of smaller math questions that helps solve the problem, progressively getting closer to the answer. Do not mention the solution in the hint.")


hint_generator = dspy.ChainOfThought(MathHintsGenerator)

## Evaluating `MathHintsGenerator`

We'd like to evaluate the performance of our DSPy hint generator, i.e. create a metric that evaluates the quality of generated hints. We do this by creating an 'LLM-as-Judge' metric, where we create a different DSPy module that takes a hint as input, and produces a numerical score as output.

The following `HintEvaluator` is the signature for our metric. The metric is based off of chapter 1 of George Pólya's *How to Solve It,* a famous text on mathematics teaching and problem solving. Chapter 1: *In the Classroom* outlines several techniques for giving effective hints to a math student, so that it 1. helps the student solve the question at hand, and 2. to develop the student's ability so that he may solve future problems by himself.

The hint is evaluated on 6 different criteria:
- Does the hint guide toward solution without revealing the answer?
- Does the hint use questions to prompt thinking rather than just stating facts?
- Does the hint draw attention to relevant data, unknowns, or conditions?
- Does the hint suggest a concrete, actionable next step?
- Is the hint at appropriate difficulty level for this stage?
- If not first hint, does it build logically on previous hints?

In [5]:
# Based off of George Polya's "How to Solve It," Chapter 1.

class HintEvaluator(dspy.Signature):
    """Evaluate the quality of a sequence of mathematical hints.
    
    A good hint should:
    1. Guide without giving away the answer
    2. Ask questions rather than make statements when possible
    3. Draw attention to relevant aspects of the problem
    4. Connect to student's existing knowledge
    5. Suggest a concrete, actionable step
    6. Be appropriately sized (not too broad, not too specific)
    """
    
    problem: str = dspy.InputField(desc="The math problem the student is trying to solve")
    hint: str = dspy.InputField(desc="The hint that was generated")
    previous_hints: list[str] = dspy.InputField(desc="Previous hints given, if any")
    
    # Evaluation criteria
    guides_without_revealing: int = dspy.OutputField(desc="Score 1-5: Does the hint guide toward solution without revealing the answer?")
    uses_questions: int = dspy.OutputField(desc="Score 1-5: Does the hint use questions to prompt thinking rather than just stating facts?")
    draws_attention: int = dspy.OutputField(desc="Score 1-5: Does the hint draw attention to relevant data, unknowns, or conditions?")
    suggests_action: int = dspy.OutputField(desc="Score 1-5: Does the hint suggest a concrete, actionable next step?")
    appropriate_difficulty: int = dspy.OutputField(desc="Score 1-5: Is the hint at appropriate difficulty level for this stage?")
    builds_on_previous: int = dspy.OutputField(desc="Score 1-5: If not first hint, does it build logically on previous hints?")

Our metric takes in a single hint as input, rather than a list of hints. To evaluate a sequence of hints, we loop through the sequence and evaluate each hint individually. This is implemented in the `forward` method below. The metric also keeps track of scores for each individual criterion above.

In [6]:
class HintMetric(dspy.Module):
    """Metric for evaluating hint quality."""
    
    def __init__(self):
        self.evaluator = dspy.ChainOfThought(HintEvaluator)
    
    def forward(self, example: dspy.Example, pred: dspy.Prediction, trace=None) -> dspy.Prediction:
        hints = pred.hints
        overall_scores = []
        guides_without_revealing_scores = []
        uses_questions_scores = []
        draws_attention_scores = []
        suggests_action_scores = []
        appropriate_difficulty_scores = []
        builds_on_previous_scores = []

        # Evaluates each hint in the sequence individually, then averages the overall scores
        for i in range(len(hints)):
            eval_result = self.evaluator(
                problem=example.problem,
                hint=hints[i],
                previous_hints=hints[:i]
            )

            # Extract overall score
            scores = [
                eval_result.guides_without_revealing,
                eval_result.uses_questions,
                eval_result.draws_attention,
                eval_result.suggests_action,
                eval_result.appropriate_difficulty,
                eval_result.builds_on_previous
            ]
            overall_score = sum(scores) / len(scores)  # Average score out of 5
            overall_scores.append(overall_score / 5.0)  # Normalize to [0, 1]

            # Extract individual criteria scores
            guides_without_revealing_scores.append(eval_result.guides_without_revealing / 5.0)
            uses_questions_scores.append(eval_result.uses_questions / 5.0)
            draws_attention_scores.append(eval_result.draws_attention / 5.0)
            suggests_action_scores.append(eval_result.suggests_action / 5.0)
            appropriate_difficulty_scores.append(eval_result.appropriate_difficulty / 5.0)
            builds_on_previous_scores.append(eval_result.builds_on_previous / 5.0)

        # Aggregate all scores
        score = sum(overall_scores) / len(overall_scores) if overall_scores else 0.0
        guides_without_revealing_score = sum(guides_without_revealing_scores) / len(guides_without_revealing_scores) if guides_without_revealing_scores else 0.0
        uses_questions_score = sum(uses_questions_scores) / len(uses_questions_scores) if uses_questions_scores else 0.0
        draws_attention_score = sum(draws_attention_scores) / len(draws_attention_scores) if draws_attention_scores else 0.0
        suggests_action_score = sum(suggests_action_scores) / len(suggests_action_scores) if suggests_action_scores else 0.0
        appropriate_difficulty_score = sum(appropriate_difficulty_scores) / len(appropriate_difficulty_scores) if appropriate_difficulty_scores else 0.0
        builds_on_previous_score = sum(builds_on_previous_scores) / len(builds_on_previous_scores) if builds_on_previous_scores else 0.0

        return dspy.Prediction(
            score=score,
            guides_without_revealing=guides_without_revealing_score,
            uses_questions=uses_questions_score,
            draws_attention=draws_attention_score,
            suggests_action=suggests_action_score,
            appropriate_difficulty=appropriate_difficulty_score,
            builds_on_previous=builds_on_previous_score
        )


hint_metric = HintMetric()

Let's evaluate HINTy with this metric. We'll log this evaluation run in MLflow.

In [7]:
with mlflow.start_run(run_name="HINTy_evaluation"):
    evaluate = dspy.Evaluate(
        devset=dataset,
        metric=hint_metric,
        num_threads=24,
        display_progress=True,
    )

    # Evaluate the program as usual
    result = evaluate(hint_generator)

    # Log the aggregated score
    mlflow.log_metric("accuracy", result.score)

    # Log the detailed evaluation results as a table
    mlflow.log_table(
        {
            "Problem": [example.problem for example in dataset],
            "Answer": [example.answer for example in dataset],
            "Predicted Hints": [output[1].hints for output in result.results],
            "Reasoning": [output[1].reasoning for output in result.results],
            "Score": [output[2].score for output in result.results],
            "Guides Without Revealing": [output[2].guides_without_revealing for output in result.results],
            "Uses Questions": [output[2].uses_questions for output in result.results],
            "Draws Attention": [output[2].draws_attention for output in result.results],
            "Suggests Action": [output[2].suggests_action for output in result.results],
            "Appropriate Difficulty": [output[2].appropriate_difficulty for output in result.results],
            "Builds on Previous": [output[2].builds_on_previous for output in result.results],
        },
        artifact_file="results.json",
    )

Average Metric: 95.06 / 100 (95.1%): : 102it [08:56,  5.26s/it]                      

2026/01/06 16:32:40 INFO dspy.evaluate.evaluate: Average Metric: 95.06486772486767 / 100 (95.1%)



🏃 View run HINTy_evaluation at: http://127.0.0.1:5000/#/experiments/4/runs/164f966d80a04883b56f409594f0f161
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[Trace(trace_id=tr-5e7ab1837d38b7a33c8b7e71987e1b1e), Trace(trace_id=tr-b6414fedbf1bd817d5855c3254384fac), Trace(trace_id=tr-dfdbb91bf9537063c2a8775349811296), Trace(trace_id=tr-e4d565ee9cb3e59c945eaad44ee19919), Trace(trace_id=tr-d811cd84a9421d01fa7dab5f98c4227f), Trace(trace_id=tr-5892b424a0928ecf26e1f7184fea5be1), Trace(trace_id=tr-7790aa0efa391b9bbc4bc26f5bd2d387), Trace(trace_id=tr-1b416559f87cf63beeedb08dc74e8546), Trace(trace_id=tr-f8ea0d5a8d95742d9c8b0b4b39ca00cb), Trace(trace_id=tr-b4d7eab04ab132d9bc50733da4e096e1)]

Using our LLM-as-Judge metric, our DSPy hint generation program scores **95.06%**: very high! Results for this evaluation run can be found in `experiments/results.json`.